In [0]:
import mlflow
import mlflow.sklearn
import pandas as pd

mlflow.set_registry_uri("databricks-uc")

feature_cols = [
    "orders_hist",
    "items_hist",
    "revenue_hist_inr",
    "avg_order_value_hist_inr",
    "avg_discount_percent_hist",
    "coupon_orders_hist",
    "distinct_channels_hist",
    "recency_days"
]

model = mlflow.sklearn.load_model("models:/ecommerce.ml.purchase_propensity_model/1")

score_pdf = spark.table("ecommerce.ml.customer_features").toPandas()
score_pdf["purchase_score_next_30d"] = model.predict_proba(score_pdf[feature_cols])[:, 1]

scores_sdf = spark.createDataFrame(
    score_pdf[["customer_id", "purchase_score_next_30d"]]
)

(
    scores_sdf.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("ecommerce.ml.customer_scores")
)

In [0]:
spark.sql("""
SELECT *
FROM ecommerce.ml.customer_scores
ORDER BY purchase_score_next_30d DESC
LIMIT 20
""").display()